In [ ]:
import pandas as pd

# Load both CSV files
df1 = pd.read_csv('matches.csv')
df2 = pd.read_csv('deliveries.csv')
print(df1.info())  
print(df2.info())   
print(df1.head())  # Check data types and missing values
print(df2.head())  # View first few rows


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

In [5]:
merged_df=pd.merge(df1,df2,left_on='id',right_on='match_id',how='right')
print(merged_df.head())

       id   season       city        date match_type player_of_match  \
0  335982  2007/08  Bangalore  2008-04-18     League     BB McCullum   
1  335982  2007/08  Bangalore  2008-04-18     League     BB McCullum   
2  335982  2007/08  Bangalore  2008-04-18     League     BB McCullum   
3  335982  2007/08  Bangalore  2008-04-18     League     BB McCullum   
4  335982  2007/08  Bangalore  2008-04-18     League     BB McCullum   

                   venue                        team1                  team2  \
0  M Chinnaswamy Stadium  Royal Challengers Bangalore  Kolkata Knight Riders   
1  M Chinnaswamy Stadium  Royal Challengers Bangalore  Kolkata Knight Riders   
2  M Chinnaswamy Stadium  Royal Challengers Bangalore  Kolkata Knight Riders   
3  M Chinnaswamy Stadium  Royal Challengers Bangalore  Kolkata Knight Riders   
4  M Chinnaswamy Stadium  Royal Challengers Bangalore  Kolkata Knight Riders   

                   toss_winner  ...   bowler  non_striker batsman_runs  \
0  Royal Cha

In [6]:
merged_df["date"] = pd.to_datetime(merged_df["date"], errors="coerce")


In [7]:
merged_df["result_margin"] = merged_df["result_margin"].fillna(0).astype(int)


In [8]:
categorical_cols = ["team1", "team2", "toss_winner", "toss_decision", "winner", "result"]
merged_df[categorical_cols] = merged_df[categorical_cols].fillna("Unknown")


In [9]:
merged_df =merged_df.sort_values(by=["date", "inning", "over", "ball"])


In [10]:
merged_df['team1']=merged_df['team1']
merged_df['team2']=merged_df['team2']
print("\n renamed Teams:")
print(merged_df[['team1','team2']].head())


 renamed Teams:
                         team1                  team2
0  Royal Challengers Bangalore  Kolkata Knight Riders
1  Royal Challengers Bangalore  Kolkata Knight Riders
2  Royal Challengers Bangalore  Kolkata Knight Riders
3  Royal Challengers Bangalore  Kolkata Knight Riders
4  Royal Challengers Bangalore  Kolkata Knight Riders


In [11]:
merged_df['Cumulative_runs']=merged_df.groupby(["id","inning"])['total_runs'].cumsum()
print("\nCumulative runs")
print(merged_df[['id','inning','Cumulative_runs']].head())


Cumulative runs
       id  inning  Cumulative_runs
0  335982       1                1
1  335982       1                1
2  335982       1                2
3  335982       1                2
4  335982       1                2


In [12]:
merged_df['Cumulative_wickets']=merged_df.groupby(["id","inning"])['player_dismissed'].transform(lambda x: x.notna().sum())
print("\nCumulative Wickets")
print(merged_df[['id','inning','Cumulative_wickets']].head())


Cumulative Wickets
       id  inning  Cumulative_wickets
0  335982       1                   3
1  335982       1                   3
2  335982       1                   3
3  335982       1                   3
4  335982       1                   3


In [13]:
merged_df['current_run_rate']=merged_df.groupby(["id","inning"])['total_runs'].cumsum()/(merged_df.groupby(["id","inning"])['over'].cummax()+ 0.1)
print("\ncurrent run rate")
print(merged_df[['id','inning','current_run_rate']].head())


current run rate
       id  inning  current_run_rate
0  335982       1              10.0
1  335982       1              10.0
2  335982       1              20.0
3  335982       1              20.0
4  335982       1              20.0


In [14]:
merged_df['overs_completed']=merged_df.groupby(["id","inning"])['over'].cummax()
print("\novers completed")
print(merged_df[['id','inning','overs_completed']].head())


overs completed
       id  inning  overs_completed
0  335982       1                0
1  335982       1                0
2  335982       1                0
3  335982       1                0
4  335982       1                0


In [15]:
merged_df.to_csv("ipl_data_with_new_features.csv",index=False)
print("\n new dataset ipl_data_with_new_features.csv completed")



 new dataset ipl_data_with_new_features.csv completed


In [16]:
new_df=pd.read_csv("ipl_data_with_new_features.csv")
new_df.head()

C:\Users\jyoth\AppData\Local\Temp\ipykernel_11372\196864376.py:1: DtypeWarning: Columns (1,17) have mixed types. Specify dtype option on import or set low_memory=False.
  new_df=pd.read_csv("ipl_data_with_new_features.csv")


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,Cumulative_runs,Cumulative_wickets,current_run_rate,overs_completed
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,1,legbyes,0,NaN,NaN,NaN,1,3,10.0,0
1,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,0,NaN,0,NaN,NaN,NaN,1,3,10.0,0
2,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,1,wides,0,NaN,NaN,NaN,2,3,20.0,0
3,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,0,NaN,0,NaN,NaN,NaN,2,3,20.0,0
4,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,0,NaN,0,NaN,NaN,NaN,2,3,20.0,0


In [26]:
from sklearn.preprocessing import LabelEncoder
# Apply Label Encoding to categorical columns
label_encoders = {}
for col in ["batting_team", "bowling_team", "venue"]:
    label_encoders[col] = LabelEncoder()
    merged_df[col + "_encoded"] = label_encoders[col].fit_transform(merged_df[col])
    
# Display the encoded columns
print(merged_df[["batting_team_encoded", "bowling_team_encoded", "venue_encoded"]].head())


   batting_team_encoded  bowling_team_encoded  venue_encoded
0                     8                    16             23
1                     8                    16             23
2                     8                    16             23
3                     8                    16             23
4                     8                    16             23


In [25]:
print(merged_df.columns)

Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2', 'match_id', 'inning', 'batting_team',
       'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker',
       'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket',
       'player_dismissed', 'dismissal_kind', 'fielder', 'Cumulative_runs',
       'Cumulative_wickets', 'current_run_rate', 'overs_completed',
       'batting_team_encoded', 'bowling_team_encoded', 'venue_encoded', 'win',
       'remaining_overs'],
      dtype='object')


In [31]:
from sklearn.model_selection import train_test_split

# Create 'win' column (1 if batting team wins, 0 otherwise)
merged_df['win'] = (merged_df['winner'] == merged_df['batting_team']).astype(int)

# Select final required features
final_df = merged_df[[
    'inning', 'Cumulative_runs', 'Cumulative_wickets', 
    'current_run_rate', 'target_runs', 
    'batting_team_encoded', 'bowling_team_encoded', 
    'venue_encoded', 'win'
]]

# Train-Test Split
X = final_df.drop('win', axis=1)
y = final_df['win']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train Set Shape:", X_train.shape)
print("Test Set Shape:", X_test.shape)
print(X_train)
print(X_test)
print(y_train)
print(y_test)

Train Set Shape: (208736, 8)
Test Set Shape: (52184, 8)
        inning  Cumulative_runs  Cumulative_wickets  current_run_rate  \
196210       2               47                   4         11.463415   
77002        2               11                  10          3.548387   
179750       2               71                   2          6.396396   
48197        2              103                   9          6.023392   
250502       1               99                   7          8.181818   
...        ...              ...                 ...               ...   
259178       2               22                   5         10.476190   
103638       1               66                   5          8.148148   
131932       2               46                   8          5.679012   
146955       2               35                  10          6.862745   
121958       2               71                  10          7.029703   

        target_runs  batting_team_encoded  bowling_team_encoded  ve

In [29]:
from sklearn.linear_model import LogisticRegression

In [34]:
print(merged_df.isnull().sum())

id                           0
season                       0
city                     12397
date                         0
match_type                   0
player_of_match            490
venue                        0
team1                        0
team2                        0
toss_winner                  0
toss_decision                0
winner                       0
result                       0
result_margin                0
target_runs                309
target_overs               309
super_over                   0
method                  257274
umpire1                      0
umpire2                      0
match_id                     0
inning                       0
batting_team                 0
bowling_team                 0
over                         0
ball                         0
batter                       0
bowler                       0
non_striker                  0
batsman_runs                 0
extra_runs                   0
total_runs                   0
extras_t

In [42]:
fl=merged_df.fillna(5)
print(fl)
print(fl.isnull().sum())

             id   season       city       date match_type player_of_match  \
0        335982  2007/08  Bangalore 2008-04-18     League     BB McCullum   
1        335982  2007/08  Bangalore 2008-04-18     League     BB McCullum   
2        335982  2007/08  Bangalore 2008-04-18     League     BB McCullum   
3        335982  2007/08  Bangalore 2008-04-18     League     BB McCullum   
4        335982  2007/08  Bangalore 2008-04-18     League     BB McCullum   
...         ...      ...        ...        ...        ...             ...   
260915  1426312     2024    Chennai 2024-05-26      Final        MA Starc   
260916  1426312     2024    Chennai 2024-05-26      Final        MA Starc   
260917  1426312     2024    Chennai 2024-05-26      Final        MA Starc   
260918  1426312     2024    Chennai 2024-05-26      Final        MA Starc   
260919  1426312     2024    Chennai 2024-05-26      Final        MA Starc   

                                           venue                        tea